# QRC embeddings — clean EuroSAT (σ = 0)

Computes the Rydberg-reservoir embeddings of the **noise-free** EuroSAT images,
for the classification arm of the QRC study. Embeddings only: no MLP is trained
and no metric is computed here.

Everything numerical is imported from `train_scripts/run_eurosat_qrc_pipeline.py`,
which carries the same dataset loading, split, PCA, scaling and reservoir as the
denoising paper (`../qrc-satellite-imagery-denoising`, frozen). Nothing is
re-implemented here, so the rows of these embeddings line up, image for image,
with the σ > 0 embeddings published in that paper's release.

**What σ = 0 means mechanically.** `apply_noise(X, 0, seed)` draws
`eps ~ N(0, 0) = 0`, so `clip(X · (1 + 0), 0, 1) = X` exactly — the PCA input is
the clean image. The PCA itself is fitted on the clean training split either way,
so only the projected data changes.

**Outputs** (under `models/eurosat_qrc/`, the same layout the pipeline script uses):

| file | shape | what |
|---|---|---|
| `sigma0.0_embeddings_d18/{train,val,test}.npy` | (n, 1368) | reservoir observables |
| `sigma0.0_embeddings_d18/pca18_{train,val,test}.npy` | (n, 18) | the classical arm's features, same rows |
| `sigma0.0_embeddings_d18/pca_evr.npy` | (100,) | explained variance ratio |
| `split_labels_seed42.npz` | (n,) ×3 | class labels + category names |

The 18 scaled PCs are saved next to the embeddings on purpose: the two arms of
the ablation have to come from *identical rows*, and deriving them twice in two
notebooks is how that silently stops being true.

**Cost.** One reservoir pass over all 27000 images. `compute_embeddings_gpu`
caches per split, so an interrupted run resumes at split granularity — rerun the
cell and the finished splits load from disk.

In [1]:
import os
import random
import sys
import time
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

import numpy as np
import tensorflow as tf
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

# The reservoir, the dataset loader and the embedding loop all live in the
# pipeline script; importing them is what keeps this notebook consistent with
# the paper instead of being a second implementation of the same physics.
sys.path.insert(0, str(Path.cwd() / "train_scripts"))
import run_eurosat_qrc_pipeline as qrc

SEED    = qrc.SEED          # 42
D_QRC   = 18                # qubits = PCA components fed to the reservoir
SIGMA   = 0.0               # this notebook is the clean run
N_COMPONENTS = qrc.N_COMPONENTS

OUT_DIR   = qrc.MODEL_DIR / f"sigma{SIGMA}_embeddings_d{D_QRC}"
LABEL_PATH = qrc.MODEL_DIR / f"split_labels_seed{SEED}.npz"
OUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
# TF32 is on by default from Ampere on and silently lowers float32 matmul
# precision -- the reservoir propagation is all matmul. Off, for reproducibility;
# the same fix went into the paper's pipeline.
tf.config.experimental.enable_tensor_float_32_execution(False)
for _g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(_g, True)

print(f"σ = {SIGMA}, d = {D_QRC}, GPU: {tf.config.list_physical_devices('GPU')}")
print(f"TF {tf.__version__} | TF32 "
      f"{tf.config.experimental.tensor_float_32_execution_enabled()}")
print(f"out: {OUT_DIR}")

σ = 0.0, d = 18, GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TF 2.21.0 | TF32 False
out: models/eurosat_qrc/sigma0.0_embeddings_d18


## Dataset and split

Identical to the paper: categories and files enumerated with `sorted()`, resized
to 64×64, scaled to [0, 1], Rec.601 grayscale, then a stratified 80/10/10 split
with `random_state=42`. Deterministic, so the row order is the paper's row order
— which is the only reason the labels below can be attached to embeddings
computed there.

`data_zip/EuroSAT_RGB.zip` is already in this repo, so nothing is downloaded;
the cell just extracts it the first time.

In [2]:
qrc.download_eurosat()                 # extracts data_zip/EuroSAT_RGB.zip if needed
X, y = qrc.load_dataset()              # (N, 64, 64) grayscale in [0, 1], labels 0..9
CATEGORIES = sorted(p.name for p in qrc.DATA_DIR.iterdir() if p.is_dir())

# Same two calls as qrc.split_dataset, but keeping the labels: the script drops
# them because denoising has no use for them, and classification is all label.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp)

assert (len(X_train), len(X_val), len(X_test)) == (21600, 2700, 2700), \
    "split sizes differ from the paper — the row order is no longer comparable"

print(f"{len(CATEGORIES)} categories: {CATEGORIES}")
print(f"train {X_train.shape}  val {X_val.shape}  test {X_test.shape}")
print("\nclass counts (train / val / test):")
for i, cat in enumerate(CATEGORIES):
    print(f"  {cat:<22}{(y_train == i).sum():>7}{(y_val == i).sum():>7}{(y_test == i).sum():>7}")

Categories: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Total: (27000, 64, 64)
10 categories: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
train (21600, 64, 64)  val (2700, 64, 64)  test (2700, 64, 64)

class counts (train / val / test):
  AnnualCrop               2400    300    300
  Forest                   2400    300    300
  HerbaceousVegetation     2400    300    300
  Highway                  2000    250    250
  Industrial               2000    250    250
  Pasture                  1600    200    200
  PermanentCrop            2000    250    250
  Residential              2400    300    300
  River                    2000    250    250
  SeaLake                  2400    300    300


## σ = 0 really is the identity

Cheap assertion rather than a comment: if the noise model ever stops being
multiplicative, this fails instead of quietly producing embeddings of something
else.

In [3]:
assert np.array_equal(qrc.apply_noise(X_train, sigma=SIGMA, seed=SEED), X_train)
assert np.array_equal(qrc.apply_noise(X_test,  sigma=SIGMA, seed=SEED + 2), X_test)
print("σ = 0 leaves every pixel untouched — the reservoir will see clean images")

σ = 0 leaves every pixel untouched — the reservoir will see clean images


## PCA(100) → 18 components → [0, 1]

PCA is fitted on the **clean training split** (as in the paper), the first 18
components are kept, and they are min–max scaled with *training* statistics
because the reservoir maps each component onto a local detuning expecting [0, 1].

Note the scaling is per-run: these min/max come from clean PCs and differ from
the σ > 0 runs' own statistics. That is intended — each run scales with its own
training data — but it means these 18-dim features are not element-wise
comparable with the noisy runs'.

In [4]:
pca = PCA(n_components=N_COMPONENTS)
pca.fit(X_train.reshape(len(X_train), -1))
evr = pca.explained_variance_ratio_
print(f"PC1: {100 * evr[0]:.2f}%   PCs 1-{D_QRC}: {100 * evr[:D_QRC].sum():.2f}%"
      f"   all {N_COMPONENTS}: {100 * evr.sum():.2f}%")

z = {name: pca.transform(Xs.reshape(len(Xs), -1))[:, :D_QRC]
     for name, Xs in (("train", X_train), ("val", X_val), ("test", X_test))}
z_min, z_max = z["train"].min(axis=0), z["train"].max(axis=0)


def scale_z(v, eps=1e-8):
    return np.clip((v - z_min) / (z_max - z_min + eps), 0.0, 1.0).astype(np.float64)


z = {k: scale_z(v) for k, v in z.items()}
print("reservoir inputs:", {k: v.shape for k, v in z.items()},
      f"range [{min(v.min() for v in z.values()):.3f}, {max(v.max() for v in z.values()):.3f}]")

PC1: 66.91%   PCs 1-18: 81.08%   all 100: 89.83%
reservoir inputs: {'train': (21600, 18), 'val': (2700, 18), 'test': (2700, 18)} range [0.000, 1.000]


## Reservoir

18 qubits, exact state vector of dimension 2¹⁸, Ising Hamiltonian with
input-dependent local detunings, 8 quench times, observables ⟨Zᵢ⟩ and ⟨ZᵢZⱼ⟩ →
R = 1368 features. The smoke check compares the GPU Chebyshev propagation
against the exact CPU `expm_multiply` reference on 5 images; it is the guard
that the fast path is still computing the same physics.

In [5]:
cfg = qrc.QRCConfig(n_qubits=D_QRC)
reservoir = qrc.RydbergReservoir(cfg)
tf_reservoir = qrc.TFReservoir(reservoir, cfg)

# Measured on this machine: the exact CPU reference costs ~64 s per image, so
# this cell runs for ~3 min. The paper used 5 images; 3 is the same check.
N_SMOKE = 3
cpu_ref = np.array([reservoir.transform_one(v) for v in z["train"][:N_SMOKE]])
max_err = np.abs(tf_reservoir.embed_batch(z["train"][:N_SMOKE]) - cpu_ref).max()
print(f"GPU vs CPU smoke — max abs error: {max_err:.2e}")
assert max_err < 5e-3, "GPU reservoir disagrees with the CPU reference beyond fp32 tolerance"

RydbergReservoir: d=18, dim=262144, obs=171 (18 Z + 153 ZZ), embed_dim=1368
GPU vs CPU smoke — max abs error: 3.83e-06


## Embeddings

The long one. Each split is cached to its own `.npy`, so this cell is safe to
interrupt and rerun: finished splits load from disk, and only the missing one is
recomputed. The loop prints a rate and an ETA every 20 batches.

In [6]:
CACHE = {s: OUT_DIR / f"{s}.npy" for s in ("train", "val", "test")}

t0 = time.perf_counter()
# Measured here: ~0.60 s/image at batch 32 on the RTX 3050, so ~4.5 h for all
# 27000 (test and val are ~27 min each, train ~3.6 h). BATCH sets the GPU batch;
# the state vector is 2^18 complex64 = 2 MB per image, so 32 is ~67 MB of states
# and there is room to raise it on a 4 GB card.
BATCH = 32

R = {s: qrc.compute_embeddings_gpu(tf_reservoir, z[s], CACHE[s], batch_size=BATCH)
     for s in ("test", "val", "train")}   # small splits first: fail fast, cheap
print(f"\ntotal {(time.perf_counter() - t0) / 60:.1f} min")
print("embeddings:", {k: v.shape for k, v in R.items()})

Computing 2700 embeddings on GPU (batch=32) ...
  32/2700  (0.564s/img, eta 25.1 min)
  672/2700  (0.569s/img, eta 19.2 min)
  1312/2700  (0.568s/img, eta 13.1 min)
  1952/2700  (0.568s/img, eta 7.1 min)
  2592/2700  (0.568s/img, eta 1.0 min)
Done in 25.6 min -> saved to models/eurosat_qrc/sigma0.0_embeddings_d18/test.npy
Computing 2700 embeddings on GPU (batch=32) ...
  32/2700  (0.560s/img, eta 24.9 min)
  672/2700  (0.566s/img, eta 19.1 min)
  1312/2700  (0.567s/img, eta 13.1 min)
  1952/2700  (0.567s/img, eta 7.1 min)
  2592/2700  (0.567s/img, eta 1.0 min)
Done in 25.5 min -> saved to models/eurosat_qrc/sigma0.0_embeddings_d18/val.npy
Computing 21600 embeddings on GPU (batch=32) ...
  32/21600  (0.568s/img, eta 204.1 min)
  672/21600  (0.570s/img, eta 198.7 min)
  1312/21600  (0.570s/img, eta 192.6 min)
  1952/21600  (0.570s/img, eta 186.5 min)
  2592/21600  (0.569s/img, eta 180.3 min)
  3232/21600  (0.569s/img, eta 174.2 min)
  3872/21600  (0.569s/img, eta 168.1 min)
  4512/21600 

## Save the rest of the arm

The embeddings alone are not a usable artefact: the classification experiment
needs the labels (same rows) and the 18 scaled PCs (the classical arm of the
ablation). Saved together here so no downstream notebook has to re-derive them.

In [7]:
for s in ("train", "val", "test"):
    np.save(OUT_DIR / f"pca18_{s}.npy", z[s])
np.save(OUT_DIR / "pca_evr.npy", evr)

np.savez(LABEL_PATH,
         y_train=y_train, y_val=y_val, y_test=y_test,
         categories=np.array(CATEGORIES), seed=SEED)

print(f"saved to {OUT_DIR}:")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name:<22}{p.stat().st_size / 1e6:>9.1f} MB")
print(f"\nsaved {LABEL_PATH}")

# Final consistency check: embeddings, PCs and labels must agree row for row.
for s, lab in (("train", y_train), ("val", y_val), ("test", y_test)):
    assert len(R[s]) == len(z[s]) == len(lab), f"{s}: row counts disagree"
    assert R[s].shape[1] == 1368 and z[s].shape[1] == D_QRC
print("rows aligned across embeddings, PCs and labels")

saved to models/eurosat_qrc/sigma0.0_embeddings_d18:
  pca18_test.npy              0.4 MB
  pca18_train.npy             3.1 MB
  pca18_val.npy               0.4 MB
  pca_evr.npy                 0.0 MB
  test.npy                   29.5 MB
  train.npy                 236.4 MB
  val.npy                    29.5 MB

saved models/eurosat_qrc/split_labels_seed42.npz
rows aligned across embeddings, PCs and labels
